# Content Ranking Eval Pipeline

**Can a text-based quality rubric predict what HackerNews upvotes?**

Rachel McCaig (Barden) · June 2026 · [GitHub](https://github.com/rachelmccaig/content-ranking-eval)

---

This notebook walks through a model-as-judge eval pipeline that scores HackerNews stories on content quality dimensions, then validates those scores against real human engagement using Spearman rank correlation.

**The core question:** does writing quality predict community engagement?

**The finding:** after 8 runs and two statistically significant results at n=499, the relationship is measurably *negative* — and the reason why turns out to be the interesting part.

Cells with baked outputs show results from actual API runs. The demo cell (heuristic mode) is fully runnable without an API key.

## The rubric

Four dimensions scored 1–10 by Claude Opus (or Haiku for the large runs):

| Dimension | Weight | What it measures |
|---|---|---|
| `title_clarity` | 20% | Is the title clear and understandable? |
| `specificity` | 25% | Specific and concrete vs. vague or generic? |
| `topic_novelty` | 30% | Fresh or non-obvious? Penalizes rehashed takes. |
| `information_density` | 25% | Real signal vs. clickbait? |

**The dimension that was removed:** `engagement_potential` was in v1. It's a circular predictor — using "how likely is this to get engagement" to predict engagement encodes the answer and makes the other dimensions irrelevant. Caught in the first run, replaced with `topic_novelty`.

**The dimension that was considered and rejected:** `community_identity` — scoring whether content would resonate with the HN community. Ruled out because it can't be grounded in text alone. To score it, the judge would need to know who Fabrice Bellard is, why HN cares about him, and what the current community temperature is. Asking the model to estimate that from a title produces hallucinated scores that look plausible but corrupt the correlation in ways that are hard to diagnose.

**Ground truth:** composite engagement score = `hn_score + (comments × 2)`. Comments require active intent; upvotes are passive. Chosen interaction signals are more valuable for ranking than impression-level signals.

In [1]:
WEIGHTS = {
    "title_clarity":       0.20,
    "specificity":         0.25,
    "topic_novelty":       0.30,
    "information_density": 0.25,
}
COMMENT_WEIGHT = 2

def composite(scores: dict) -> float:
    return sum(scores[dim] * weight for dim, weight in WEIGHTS.items())

print("Rubric weights:")
for dim, weight in WEIGHTS.items():
    print(f"  {dim:<22} {weight:.0%}")
print(f"\nGround truth: hn_score + (comments × {COMMENT_WEIGHT})")

Rubric weights:
  title_clarity          20%
  specificity            25%
  topic_novelty          30%
  information_density    25%

Ground truth: hn_score + (comments × 2)


## Demo: heuristic scorer (no API key needed)

To show the pipeline shape without an API call, a rule-based heuristic scorer uses title length, colons, numbers, and question marks as proxies for the rubric dimensions.

The examples below were chosen to illustrate the construct gap: titles the judge reliably rates high vs. titles HN reliably rewards.

In [2]:
def score_heuristic(title: str) -> dict:
    words = title.split()
    wc = len(words)
    has_number = any(w.replace(",", "").replace(".", "").isdigit() for w in words)
    has_colon  = ":" in title
    is_q       = title.endswith("?")
    return {
        "title_clarity":       min(10, max(1, 5 + (2 if has_colon else 0) + (1 if wc < 12 else -1))),
        "specificity":         min(10, max(1, 4 + (3 if has_number else 0) + (2 if has_colon else 0))),
        "topic_novelty":       min(10, max(1, 5 + (2 if not is_q else 0) + (1 if has_number else 0))),
        "information_density": min(10, max(1, min(wc, 10) - (2 if is_q else 0))),
    }

examples = [
    "Building a Custom Memory Allocator in C++: A Complete Guide",
    "A Deep Dive into PostgreSQL's VACUUM Process",
    "Running local models is good now",
    "Ask HN: What are you working on?",
    "Nipkow Disk Mechanical TV Simulator",
    "Iroh 1.0",
]

print("Heuristic scores for representative titles:\n")
scored = []
for title in examples:
    s = score_heuristic(title)
    c = composite(s)
    scored.append((c, title))
    print(f"Title: {title}")
    print(f"  clarity={s['title_clarity']}  specificity={s['specificity']}  "
          f"novelty={s['topic_novelty']}  density={s['information_density']}   "
          f"→ composite {c:.2f}\n")

scored.sort(reverse=True)
print("Judge ranking (heuristic):")
for i, (c, t) in enumerate(scored, 1):
    print(f"  #{i}  [{c:.2f}]  {t}")

print("\nActual HN engagement ranking (from live runs):")
hn_order = [
    ("Iroh 1.0",                                          "← judge #4"),
    ("Ask HN: What are you working on?",                  "← judge #3"),
    ("Running local models is good now",                  "← judge #5"),
    ("Nipkow Disk Mechanical TV Simulator",               "← judge #6"),
    ("A Deep Dive into PostgreSQL's VACUUM Process",      "← judge #2"),
    ("Building a Custom Memory Allocator in C++...",      "← judge #1"),
]
for i, (t, note) in enumerate(hn_order, 1):
    print(f"  #{i}  {t:<52} {note}")
print("\nThe judge and HN are nearly reversed on this set.")

Heuristic scores for representative titles:

Title: Building a Custom Memory Allocator in C++: A Complete Guide
  clarity=8  specificity=6  novelty=7  density=10  → composite 7.70

Title: A Deep Dive into PostgreSQL's VACUUM Process
  clarity=6  specificity=4  novelty=7  density=7   → composite 6.05

Title: Running local models is good now
  clarity=6  specificity=4  novelty=7  density=6   → composite 5.80

Title: Ask HN: What are you working on?
  clarity=8  specificity=6  novelty=5  density=5   → composite 5.85

Title: Nipkow Disk Mechanical TV Simulator
  clarity=6  specificity=4  novelty=7  density=5   → composite 5.55

Title: Iroh 1.0
  clarity=6  specificity=7  novelty=8  density=2   → composite 5.85

Judge ranking (heuristic):
  #1  [7.70]  Building a Custom Memory Allocator in C++: A Complete Guide
  #2  [6.05]  A Deep Dive into PostgreSQL's VACUUM Process
  #3  [5.85]  Ask HN: What are you working on?
  #4  [5.85]  Iroh 1.0
  #5  [5.80]  Running local models is good now
  #6  

## Iteration history

**v1** — 4 dimensions: title clarity, specificity, `engagement_potential`, information density.

Problem caught immediately: `engagement_potential` is circular. Predicting engagement with a dimension that already encodes engagement likelihood makes the other dimensions irrelevant. Removed.

**v2** — Replaced `engagement_potential` with `topic_novelty`. Added `author_karma` and `article_age_hours` as metadata signals. Switched to Opus.

Note on author karma: it turned out to be noisy. "Every Frame a Painting" posted from a domain (`tonsky.me`) that carries community trust — the HN account karma doesn't capture that. Domain-level historical engagement would be a cleaner signal.

**v3** — Updated ground truth from raw upvote score to composite engagement: `hn_score + (comments × 2)`. Comments require active intent; upvotes are passive. Added `title_word_count` as a signal following feedback that title length might matter.

The correlation continued to be negative across all runs. After diagnosing the pattern, it became clear this isn't a bug or a tuning problem — it's a construct validity problem.

In [3]:
results = [
    ("v1 heuristic", "—",     50,  -0.245, None,  "Rule-based scorer, no API"),
    ("v1 LLM",       "Opus",  50,  -0.021, None,  "4 dims incl. engagement_potential (circular)"),
    ("v2 LLM",       "Opus",  50,  -0.099, None,  "Replaced with topic_novelty, added author karma"),
    ("v2 LLM",       "Opus",  50,  -0.172, None,  "Same day, different stories"),
    ("v3 LLM",       "Opus",  50,  -0.234, None,  "Ground truth → composite engagement"),
    ("v3 LLM",       "Opus",  50,  -0.159, 0.273, "Added title_word_count signal"),
    ("v3 LLM",       "Haiku", 499, -0.113, 0.012, "First statistically significant result"),
    ("v3 LLM",       "Haiku", 499, -0.104, 0.022, "Second significant result — confirmed"),
]

print("Results across all runs:\n")
print(f"{'Run':<16} {'Model':<8} {'N':<6} {'Corr':<9} {'p':<8} {'Notes'}")
print("─" * 87)
for run, model, n, corr, p, notes in results:
    sig = "*" if p is not None and p < 0.05 else " "
    p_str = f"{p:.3f}" if p is not None else "—"
    print(f"{run:<16} {model:<8} {n:<6} {corr:<9.3f} {p_str:<8} {sig}{notes}")

print("\n* p < 0.05")
print("\nSwitching from Opus to Haiku produced no meaningful change in direction or magnitude.")
print("The construct gap is structural — more capable models don't close it; better signals would.")

Results across all runs:

Run              Model    N      Corr      p        Notes
───────────────────────────────────────────────────────────────────────────────────────────
v1 heuristic     —        50    -0.245    —        Rule-based scorer, no API
v1 LLM           Opus     50    -0.021    —        4 dims incl. engagement_potential (circular)
v2 LLM           Opus     50    -0.099    —        Replaced with topic_novelty, added author karma
v2 LLM           Opus     50    -0.172    —        Same day, different stories
v3 LLM           Opus     50    -0.234    —        Ground truth → composite engagement
v3 LLM           Opus     50    -0.159    0.273    Added title_word_count signal
v3 LLM           Haiku   499   -0.113    0.012  * First statistically significant result
v3 LLM           Haiku   499   -0.104    0.022  * Second significant result — confirmed

* p < 0.05

Switching from Opus to Haiku produced no meaningful change in direction or magnitude.
The construct gap is structur

In [4]:
disagreements = [
    {
        "title":   "Claude Fable 5",
        "judge":   473, "eng": 7, "gap": 466,
        "why":     ["Product launch for a model with massive community following.",
                    "Judge sees two words. HN sees a community event."],
    },
    {
        "title":   "Iroh 1.0",
        "judge":   465, "eng": 16, "gap": 449,
        "why":     ["Version bump for a project with a devoted community.",
                    "Same pattern — judge sees an uninformative title."],
    },
    {
        "title":   "Ask HN: What are you working on? (June 2026)",
        "judge":   454, "eng": 6, "gap": 448,
        "why":     ["Monthly ritual. Judge correctly scores it low-info.",
                    "HN shows up anyway. Community behavior, not content quality."],
    },
    {
        "title":   "I admire Fabrice Bellard",
        "judge":   450, "eng": 3, "gap": 447,
        "why":     ["Hero worship. Judge correctly penalizes it for low information",
                    "density. HN does not care."],
    },
    {
        "title":   "How to earn a billion dollars",
        "judge":   361, "eng": 5, "gap": 356,
        "why":     ["paulgraham.com. Judge correctly flags it as vague and",
                    "potentially clickbait. Judge can't see the domain authority."],
    },
]

print("Biggest disagreements — n=499 run (Haiku, Spearman -0.104, p=0.022)")
print("═" * 79)
for d in disagreements:
    print(f"\nJudge underrated by {d['gap']} positions")
    print(f"  Title:       {d['title']}")
    print(f"  Judge #{d['judge']}   Engagement #{d['eng']}")
    print(f"  Why:         {d['why'][0]}")
    print(f"               {d['why'][1]}")

print("\n" + "═" * 79)
print("These same titles appear as top disagreements across nearly every run.")
print("At n=499 they are not edge cases — they are a taxonomy.")

Biggest disagreements — n=499 run (Haiku, Spearman -0.104, p=0.022)
═══════════════════════════════════════════════════════════════════════════════

Judge underrated by 466 positions
  Title:       Claude Fable 5
  Judge #473   Engagement #7
  Why:         Product launch for a model with massive community following.
               Judge sees two words. HN sees a community event.

Judge underrated by 449 positions
  Title:       Iroh 1.0
  Judge #465   Engagement #16
  Why:         Version bump for a project with a devoted community.
               Same pattern — judge sees an uninformative title.

Judge underrated by 448 positions
  Title:       Ask HN: What are you working on? (June 2026)
  Judge #454   Engagement #6
  Why:         Monthly ritual. Judge correctly scores it low-info.
               HN shows up anyway. Community behavior, not content quality.

Judge underrated by 447 positions
  Title:       I admire Fabrice Bellard
  Judge #450   Engagement #3
  Why:         Hero worsh

## What this means

The negative correlation is confirmed — but this is a **construct validity problem, not a noise problem**. The judge is consistently measuring something real; it's just measuring a different construct than HN engagement rewards.

Three distinct failure modes surfaced across 8 runs:

**1. Community resonance** — Short titles for projects with cult followings ("Iroh 1.0", "Claude Fable 5"). The judge sees an uninformative title. HN sees a community event. This signal requires knowledge of which projects have devoted followings — behavioral data, not text data.

**2. Breaking news cycles** — The judge has no concept of topic momentum. A story that's the 3rd piece of coverage on an active news cycle gets penalized for a sparse title. Three of the top engagement stories in one run were all about the same Claude Fable 5 launch. The judge scored all three near the bottom.

**3. Domain authority** — `paulgraham.com` attached to a vague title is a quality guarantee to HN. The judge sees the words; HN sees the source. The same is true for `tonsky.me`, `danluu.com`, and similar high-trust personal domains.

**What this implies for production ranking:**

The three failure modes all share a property: they require behavioral infrastructure that text evals can't replicate. Community following, topic trend data, and domain authority all live in a platform's social graph — not in the content itself.

A text-based quality signal and an engagement-based ranking signal are measuring different things. That's not a failure of the eval — it's a finding. The eval layer exists precisely to maintain a quality bar independent of what the engagement signal is doing. Optimizing a text rubric against raw engagement would produce a signal that amplifies whatever already performs, including low-quality viral content.

**The next lever isn't more prompt tuning.** It's behavioral features: historical author/domain engagement rates, topic trend signals, community interaction history. The simplest first step would be counting how many times a domain or author has landed in the HN top 100 in the last 30 days — a single API-backed lookup that would capture most of the "Iroh 1.0" class of misses.

## Cross-model validation and v4

After the core iteration was complete, the v3 script was submitted to Codex for an independent engineering review. A second model validated that review before any changes were accepted.

The engineering review confirmed the analytical findings — the biggest methodological concern it flagged was that engagement is affected by timing, author reputation, and HN's own ranking algorithm, not just content quality. That's the construct validity problem this notebook describes.

Engineering improvements from that review were implemented in v4: scipy Spearman with tie-aware fallback, LLM output validation, retry/backoff, checkpointing for long runs, age-adjusted engagement normalization.

**The most interesting suggestion from the cross-model review:**

Instead of sampling `topstories` (stories that already won HN's ranking), score `newstories` at T0 and collect engagement 6–24 hours later. This removes a feedback loop: HN's ranking algorithm affects which stories get seen, which affects how much engagement they accumulate, which contaminates the ground truth. A T0 design would test whether the judge can predict future engagement from content alone, before the platform's own signal intervenes. That's the v5 direction.

---

*Built with Claude (Opus and Haiku), validated with Codex. The analytical iteration — catching the circular predictor, the composite engagement ground truth design, the construct validity diagnosis — happened across 8 runs before the engineering review.*